In [23]:
import json
import requests
import pyotp
from urllib import parse
import sys
from fyers_apiv3 import fyersModel
import os

FY_ID = "XT07804"  # Your fyers ID
APP_ID_TYPE = "2"  # Keep default as 2, It denotes web login
TOTP_KEY = "ADD5SCDJLI46IPNAUVZJZ2XJLWBJCSXX"  # TOTP secret is generated when we enable 2Factor TOTP from myaccount portal
PIN = "1997"  # User pin for fyers account
APP_ID = "63TGM29OEI"  # App ID from myapi dashboard is in the form appId-appType. Example - EGNI8CE27Q-100, In this code EGNI8CE27Q will be APP_ID and 100 will be the APP_TYPE
REDIRECT_URI = "https://localhost:8000"  # Redirect url from the app.
APP_TYPE = "100"
APP_ID_HASH = "3827b729395e684c1510fdf936c786cc3f57f982fff83f1a3379adbb6bcd9242"  # SHA-256 hash of appId-appType:appSecret

# API endpoints
BASE_URL = "https://api-t2.fyers.in/vagator/v2"
BASE_URL_2 = "https://api.fyers.in/api/v2"
URL_SEND_LOGIN_OTP = BASE_URL + "/send_login_otp"
URL_VERIFY_TOTP = BASE_URL + "/verify_otp"
URL_VERIFY_PIN = BASE_URL + "/verify_pin"
URL_TOKEN = BASE_URL_2 + "/token"
URL_VALIDATE_AUTH_CODE = BASE_URL_2 + "/validate-authcode"

SUCCESS = 1
ERROR = -1

def send_login_otp(fy_id, app_id):
    try:
        payload = {
            "fy_id": fy_id,
            "app_id": app_id
        }

        result_string = requests.post(url=URL_SEND_LOGIN_OTP, json=payload)
        if result_string.status_code != 200:
            return [ERROR, result_string.text]

        result = json.loads(result_string.text)
        request_key = result["request_key"]

        return [SUCCESS, request_key]
    
    except Exception as e:
        return [ERROR, e]
    

def generate_totp(secret):
    try:
        generated_totp = pyotp.TOTP(secret).now()
        return [SUCCESS, generated_totp]
    
    except Exception as e:
        return [ERROR, e]


def verify_totp(request_key, totp):
    try:
        payload = {
            "request_key": request_key,
            "otp": totp
        }

        result_string = requests.post(url=URL_VERIFY_TOTP, json=payload)
        if result_string.status_code != 200:
            return [ERROR, result_string.text]

        result = json.loads(result_string.text)
        request_key = result["request_key"]

        return [SUCCESS, request_key]
    
    except Exception as e:
        return [ERROR, e]


def verify_PIN(request_key, pin):
    try:
        payload = {
            "request_key": request_key,
            "identity_type": "pin",
            "identifier": pin
        }

        result_string = requests.post(url=URL_VERIFY_PIN, json=payload)
        if result_string.status_code != 200:
            return [ERROR, result_string.text]
    
        result = json.loads(result_string.text)
        access_token = result["data"]["access_token"]
        print(result)
        return [SUCCESS, access_token]
    
    except Exception as e:
        return [ERROR, e]


def token(fy_id, app_id, redirect_uri, app_type, access_token):
    try:
        payload = {
            "fyers_id": fy_id,
            "app_id": app_id,
            "redirect_uri": redirect_uri,
            "appType": app_type,
            "code_challenge": "",
            "state": "sample_state",
            "scope": "",
            "nonce": "",
            "response_type": "code",
            "create_cookie": True
        }
        headers={'Authorization': f'Bearer {access_token}'}

        result_string = requests.post(
            url=URL_TOKEN, json=payload, headers=headers
        )
        print(result_string.status_code)
        if result_string.status_code != 308:
            return [ERROR, result_string.text]

        result = json.loads(result_string.text)
        url = result["Url"]
        print(result)
        auth_code = parse.parse_qs(parse.urlparse(url).query)['auth_code'][0]

        return [SUCCESS, auth_code]
    
    except Exception as e:
        return [ERROR, e]


def validate_authcode(app_id_hash, auth_code):
    try:
        payload = {
            "grant_type": "authorization_code",
            "appIdHash": app_id_hash,
            "code": auth_code,
        }

        result_string = requests.post(url=URL_VALIDATE_AUTH_CODE, json=payload)
        if result_string.status_code != 200:
            return [ERROR, result_string.text]

        result = json.loads(result_string.text)
        access_token = result["access_token"]

        return [SUCCESS, access_token]
    
    except Exception as e:
        return [ERROR, e]

# Step 1 - Retrieve request_key from send_login_otp API
send_otp_result = send_login_otp(fy_id=FY_ID, app_id=APP_ID_TYPE)
if send_otp_result[0] != SUCCESS:
    print(f"send_login_otp failure - {send_otp_result[1]}")
    sys.exit()
else:
    print("send_login_otp success")

    # Step 2 - Generate totp
generate_totp_result = generate_totp(secret=TOTP_KEY)
if generate_totp_result[0] != SUCCESS:
    print(f"generate_totp failure - {generate_totp_result[1]}")
    sys.exit()
else:
    print("generate_totp success")

# Step 3 - Verify totp and get request key from verify_otp API
request_key = send_otp_result[1]
totp = generate_totp_result[1]
verify_totp_result = verify_totp(request_key=request_key, totp=totp)
if verify_totp_result[0] != SUCCESS:
    print(f"verify_totp_result failure - {verify_totp_result[1]}")
    sys.exit()
else:
    print("verify_totp_result success")


# Step 4 - Verify pin and send back access token
request_key_2 = verify_totp_result[1]
verify_pin_result = verify_PIN(request_key=request_key_2, pin=PIN)
if verify_pin_result[0] != SUCCESS:
    print(f"verify_pin_result failure - {verify_pin_result[1]}")
    sys.exit()
else:
    print("verify_pin_result success,")

    # Step 5 - Get auth code for API V2 App from trade access token
token_result = token(
    fy_id=FY_ID, app_id=APP_ID, redirect_uri=REDIRECT_URI, app_type=APP_TYPE,
    access_token=verify_pin_result[1]
)
print(token_result,verify_pin_result[1])
if token_result[0] != SUCCESS:
    print(f"token_result failure - {token_result[1]}")
    sys.exit()
else:
    print("token_result success")

# Step 6 - Get API V2 access token from validating auth code
auth_code = token_result[1]
validate_authcode_result = validate_authcode(
    app_id_hash=APP_ID_HASH, auth_code=auth_code
)
print(auth_code)
if token_result[0] != SUCCESS:
    print(f"validate_authcode failure - {validate_authcode_result[1]}")
    sys.exit()
else:
    print("validate_authcode success")

access_token = APP_ID + "-" + APP_TYPE + ":" + validate_authcode_result[1]

print(f"access_token - {access_token}")
fyers_log_path = os.path.join(os.getcwd(), "Fyers_logs")

client_id = validate_authcode_result[0]
fyers = fyersModel.FyersModel(client_id=client_id, is_async=False, token=validate_authcode_result[1],log_path=os.getcwd())
fyers.token=validate_authcode_result[1]

send_login_otp success
generate_totp success
verify_totp_result success
{'s': 'ok', 'code': 1004, 'message': 'Pin is Verified', 'data': {'refresh_token': 'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJodHRwczovL2xvZ2luLmZ5ZXJzLmluIiwiaWF0IjoxNzI0MTAxMzA2LCJleHAiOjE3MzE4NzczMDYsIm5iZiI6MTcyNDEwMTMwNiwiYXVkIjpbIng6MCIsIng6MSIsIng6MiIsImQ6MSJdLCJzdWIiOiJyZWZyZXNoX3Rva2VuIiwiYXRfaGFzaCI6ImdBQUFBQUJtdzdLNnVrcWFYb3p1XzlOREZBbDJXUEU2bGgza1l0cnRCRm94T1dzRkh5cHVCenBFRkNWM1BQQWoxN3pXd3htX0FLWVlWejJUZWNieFRzR2dLQkU3TG40SWxKWmtXYzlJcFJpRGo4Q2FSTkw0V1NvPSIsImRpc3BsYXlfbmFtZSI6IlRBSEVSIFNIQUJCSVIgSFVTQUlOIFNBUkFGIiwiZnlfaWQiOiJYVDA3ODA0IiwiYXBwVHlwZSI6IiIsInBvYV9mbGFnIjoiTiIsImlzTXRmRW5hYmxlZCI6Ik4iLCJpc0RkcGlFbmFibGVkIjoiTiIsImN1Z19tdGYiOiJOIn0.OHHTYYKZJt26k66TWTIaIoqlw1CwwKUp8mUvn3m6JtI', 'access_token': 'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJodHRwczovL2xvZ2luLmZ5ZXJzLmluIiwiaWF0IjoxNzI0MTAxMzA2LCJleHAiOjE3MjQxMTM4NDYsIm5iZiI6MTcyNDEwMTMwNiwiYXVkIjpbIng6MCIsIng6MSIsIng6MiIsImQ6MSJdLCJz

In [20]:
from fyers_apiv3 import fyersModel
# client_id = "YYYYYYY-100"
# access_token = "XXXXXXXXXXXXX"

# Initialize the FyersModel instance with your client_id, access_token, and enable async mode
fyers = fyersModel.FyersModel(client_id=FY_ID, token=access_token,is_async=False, log_path="")
data = {
    "symbol":"NSE:BANKNIFTY24AUG50000CE",
    "strikecount":1,
    "timestamp": ""
}
response = fyers.optionchain(data=data);
print(response)

AttributeError: 'FyersModel' object has no attribute 'optionchain'

In [26]:
import time
import pandas as pd
# Convert '1st July' and '19th August' to Unix timestamps
range_from = int(time.mktime(time.strptime("2024-08-18", "%Y-%m-%d")))
range_to = int(time.mktime(time.strptime("2024-08-19", "%Y-%m-%d")))

data = {
    "symbol": "NSE:BANKNIFTY24AUG50000CE",
    "resolution": "1",
    "date_format": "0",
    "range_from": str(range_from),  # Start date: 1st July 2024
    "range_to": str(range_to),      # End date: 19th August 2024
    "cont_flag": "1"
}

candle_data = fyers.history(data)
print(candle_data)

data = candle_data['candles']
print(data)

columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
df = pd.DataFrame(data, columns=columns)

# Converting timestamp to readable date-time format
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

# print(df)

{'candles': [[1724039100, 971.05, 989.45, 925.85, 983.65, 5550], [1724039160, 979.75, 1020, 979.75, 1017.55, 9660], [1724039220, 1015.8, 1023.5, 994, 1001.75, 7335], [1724039280, 1000, 1000, 977.05, 980, 4530], [1724039340, 980, 983.95, 959, 959, 2220], [1724039400, 951.45, 975.25, 939.55, 966, 3660], [1724039460, 962.75, 964.4, 940, 944.05, 3390], [1724039520, 947.7, 953.9, 940.2, 944, 3285], [1724039580, 940, 975, 940, 975, 1140], [1724039640, 973.15, 975, 940, 943.7, 3810], [1724039700, 938.5, 940.05, 911.2, 939.9, 4575], [1724039760, 947.15, 957.65, 941.7, 950, 3150], [1724039820, 950, 950, 925.25, 937.05, 2040], [1724039880, 932.15, 937.9, 915, 937.9, 3240], [1724039940, 937.9, 937.9, 909.05, 909.05, 2505], [1724040000, 909.05, 921, 899, 917, 4800], [1724039100, 971.05, 989.45, 925.85, 983.65, 5550], [1724039160, 979.75, 1020, 979.75, 1017.55, 9660], [1724039220, 1015.8, 1023.5, 994, 1001.75, 7335], [1724039280, 1000, 1000, 977.05, 980, 4530], [1724039340, 980, 983.95, 959, 959, 2

In [27]:
df

,timestamp,open,high,low,close,volume
0,2024-08-19 03:45:00,971.05,989.45,925.85,983.65,5550
1,2024-08-19 03:46:00,979.75,1020.00,979.75,1017.55,9660
2,2024-08-19 03:47:00,1015.80,1023.50,994.00,1001.75,7335
3,2024-08-19 03:48:00,1000.00,1000.00,977.05,980.00,4530
4,2024-08-19 03:49:00,980.00,983.95,959.00,959.00,2220
5,2024-08-19 03:50:00,951.45,975.25,939.55,966.00,3660
6,2024-08-19 03:51:00,962.75,964.40,940.00,944.05,3390
7,2024-08-19 03:52:00,947.70,953.90,940.20,944.00,3285
8,2024-08-19 03:53:00,940.00,975.00,940.00,975.00,1140
9,2024-08-19 03:54:00,973.15,975.00,940.00,943.70,3810


In [34]:
from fyers_api import fyersModel
import json
import pandas as pd
import matplotlib.pyplot as plt

client_id = "XT07804"
access_token = validate_authcode_result[1] # Ensure you have the correct access token here

# Initialize the FyersModel instance with your client_id, access_token, and disable logging
fyers = fyersModel.FyersModel(client_id=client_id, token=access_token, is_async=False)

# Function to get option chain data
def get_option_chain_data(symbol):
    data = {
        "symbols": symbol
    }
    response = fyers.quotes(data=data)
    if response['s'] == "ok":
        return response['d']
    else:
        raise Exception(f"Error fetching data: {response['message']}")

# Function to process option chain data
def process_option_chain_data(data):
    # Convert the response to a DataFrame and parse expiry dates
    df = pd.DataFrame(data)
    if 'expiry' in df.columns:
        df['expiry'] = pd.to_datetime(df['expiry'])
    return df

# Function to plot volatility term structure
def plot_volatility_term_structure(df):
    plt.figure(figsize=(10, 6))
    if 'strike' in df.columns and 'iv' in df.columns and 'expiry' in df.columns:
        for strike in df['strike'].unique():
            strike_data = df[df['strike'] == strike]
            plt.plot(strike_data['expiry'], strike_data['iv'], label=f'Strike {strike}')
    
        plt.xlabel('Expiry Date')
        plt.ylabel('Implied Volatility')
        plt.title('Volatility Term Structure')
        plt.legend()
        plt.show()
    else:
        print("Data does not contain necessary columns for plotting.")

# Main function
def main():
    symbol = 'NSE:TCS-EQ'  # Replace with your symbol
    try:
        data = get_option_chain_data(symbol)
        df = process_option_chain_data(data)
        plot_volatility_term_structure(df)
    except Exception as e:
        print(e)

if __name__ == "__main__":
    main()


ERR: logEntryFunc: quotes : [Errno 13] Permission denied: '/2024-07-13.txt'
Data does not contain necessary columns for plotting.


<Figure size 1000x600 with 0 Axes>

In [49]:
from fyers_apiv3 import fyersModel
import json
import pandas as pd
import matplotlib.pyplot as plt

client_id = "XT07804"
access_token = validate_authcode_result[1]  # Ensure you have the correct access token here

# Initialize the FyersModel instance with your client_id, access_token, and disable logging
fyers = fyersModel.FyersModel(client_id=client_id, token=access_token, is_async=False, log_path="")

# Function to get option chain data
def get_option_chain_data(symbol):
    data = {
        "symbols": symbol
    }
    response = fyers.optionchain(data=data)
    print("API Response:", response)  # Debug print to inspect the API response
    if response['s'] == "ok":
        return response['d']
    else:
        raise Exception(f"Error fetching data: {response['message']}")

# Function to process option chain data
def process_option_chain_data(data):
    # Convert the response to a DataFrame and parse expiry dates
    df = pd.DataFrame(data)
    print("Data Frame:", df)  # Debug print to inspect the DataFrame
    if 'expiry' in df.columns:
        df['expiry'] = pd.to_datetime(df['expiry'])
    return df

# Function to plot volatility term structure
def plot_volatility_term_structure(df):
    plt.figure(figsize=(10, 6))
    if 'strike' in df.columns and 'iv' in df.columns and 'expiry' in df.columns:
        for strike in df['strike'].unique():
            strike_data = df[df['strike'] == strike]
            plt.plot(strike_data['expiry'], strike_data['iv'], label=f'Strike {strike}')
    
        plt.xlabel('Expiry Date')
        plt.ylabel('Implied Volatility')
        plt.title('Volatility Term Structure')
        plt.legend()
        plt.show()
    else:
        print("Data does not contain necessary columns for plotting.")

# Main function
def main():
    symbol = 'NSE:TCS-EQ'  # Replace with your symbol
    try:
        data = get_option_chain_data(symbol)
        df = process_option_chain_data(data)
        plot_volatility_term_structure(df)
    except Exception as e:
        print(e)

if __name__ == "__main__":
    main()

'FyersModel' object has no attribute 'optionchain'


In [50]:
print(dir(fyers))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'api_logger', 'cancel_basket_orders', 'cancel_order', 'client_id', 'convert_position', 'depth', 'exit_positions', 'funds', 'generate_data_token', 'get_orders', 'get_profile', 'header', 'history', 'holdings', 'is_async', 'log_path', 'market_status', 'modify_basket_orders', 'modify_order', 'orderbook', 'place_basket_orders', 'place_order', 'positions', 'quotes', 'service', 'token', 'tradebook']
